# Windows CUA on Azure (Azure OpenAI)

Run a Computer-Using Agent on a **Windows VM** powered by **Azure OpenAI** — all on Azure.

## Prerequisites

1. **Azure CLI** installed and authenticated (`az login`)
2. An Azure subscription with quota for **Azure OpenAI** and a **Windows VM**

Everything else (OpenAI resource, API keys, VM) is created automatically by this notebook.

In [1]:
%pip install nest_asyncio

Note: you may need to restart the kernel to use updated packages.


## Step 0: Azure Configuration

Set your resource group and region. Everything else is auto-detected.

In [2]:
import subprocess, json, os, shutil

# ── Settings (edit these) ─────────────────────────────────────────────────────
RESOURCE_GROUP = "lunar-cua"            # All resources go here
LOCATION = "eastus"                     # Azure region
OPENAI_RESOURCE_NAME = "lunar-cua-ai"   # Azure AI Services resource (kind=AIServices)
OPENAI_DEPLOYMENT = "gpt-54-cua"        # Deployment name (you choose this)
OPENAI_MODEL_NAME = "gpt-5.4"           # Model (must support computer use tool)
OPENAI_MODEL_VERSION = "2026-03-05"     # Model version
VM_NAME = "cua-windows"
VM_SIZE = "Standard_D2s_v3"             # 2 vCPU, 8 GB (~$0.096/hr)
ADMIN_USER = "lunar"
ADMIN_PASSWORD = "LunarCua2024!"        # Change this!

# ── Find az CLI ───────────────────────────────────────────────────────────────
# Jupyter kernels often have a stripped PATH that misses Homebrew.
_AZ_SEARCH_PATHS = [
    "/opt/homebrew/bin",
    "/usr/local/bin",
    "/usr/bin",
    os.path.expanduser("~/.local/bin"),
]
AZ_BIN = shutil.which("az")
if AZ_BIN is None:
    for p in _AZ_SEARCH_PATHS:
        candidate = os.path.join(p, "az")
        if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            AZ_BIN = candidate
            break
if AZ_BIN is None:
    raise FileNotFoundError(
        "Azure CLI (az) not found. Install it:\n"
        "  brew install azure-cli\n"
        "  or: https://learn.microsoft.com/en-us/cli/azure/install-azure-cli"
    )

def az(args: list[str], check=True) -> dict | str:
    """Run az CLI command and return parsed JSON (or raw text)."""
    cmd = [AZ_BIN] + args + ["-o", "json"]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if check and r.returncode != 0:
        raise RuntimeError(f"az {' '.join(args[:3])} failed: {r.stderr.strip()[:300]}")
    try:
        return json.loads(r.stdout)
    except json.JSONDecodeError:
        return r.stdout.strip()

account = az(["account", "show"])
print(f"az CLI:        {AZ_BIN}")
print(f"Subscription:  {account['name']} ({account['id'][:8]}...)")
print(f"Resource Group: {RESOURCE_GROUP}")
print(f"Location:      {LOCATION}")
print(f"Model:         {OPENAI_MODEL_NAME} ({OPENAI_MODEL_VERSION})")

az CLI:        /opt/homebrew/bin/az
Subscription:  Microsoft Azure Sponsorship (3a17e97b...)
Resource Group: lunar-cua
Location:      eastus
Model:         gpt-5.4 (2026-03-05)


## Step 1: Set up Azure OpenAI resource

Creates the resource group, Azure OpenAI resource, and deployment automatically.
If they already exist, reuses them. Fetches the endpoint and API key via `az` CLI.

In [3]:
# ── 1a. Resource Group ────────────────────────────────────────────────────────
print("1/4  Resource group...", end=" ")
try:
    az(["group", "show", "--name", RESOURCE_GROUP])
    print("exists")
except RuntimeError:
    az(["group", "create", "--name", RESOURCE_GROUP, "--location", LOCATION])
    print("created")

# ── 1b. Azure AI Services Resource (kind=AIServices) ─────────────────────────
# Computer use requires AIServices kind, not OpenAI kind.
print("2/4  Azure AI Services resource...", end=" ")
try:
    oai_resource = az(["cognitiveservices", "account", "show",
                        "--name", OPENAI_RESOURCE_NAME,
                        "--resource-group", RESOURCE_GROUP])
    print(f"exists (kind={oai_resource.get('kind', '?')})")
except RuntimeError:
    print("creating (this takes ~1 minute)...")
    oai_resource = az(["cognitiveservices", "account", "create",
                        "--name", OPENAI_RESOURCE_NAME,
                        "--resource-group", RESOURCE_GROUP,
                        "--location", LOCATION,
                        "--kind", "AIServices",
                        "--sku", "S0",
                        "--custom-domain", OPENAI_RESOURCE_NAME])
    print("  created!")

# ── 1c. Get endpoint and key ─────────────────────────────────────────────────
print("3/4  Fetching credentials...", end=" ")
AZURE_OPENAI_ENDPOINT = oai_resource["properties"]["endpoint"]

keys = az(["cognitiveservices", "account", "keys", "list",
           "--name", OPENAI_RESOURCE_NAME,
           "--resource-group", RESOURCE_GROUP])
AZURE_OPENAI_API_KEY = keys["key1"]
print("done")

# ── 1d. Deployment ───────────────────────────────────────────────────────────
# gpt-5.4 supports the "computer" tool for computer use.
# Requires DataZoneStandard SKU.
print(f"4/4  Deployment '{OPENAI_DEPLOYMENT}'...", end=" ")
try:
    deployments = az(["cognitiveservices", "account", "deployment", "list",
                       "--name", OPENAI_RESOURCE_NAME,
                       "--resource-group", RESOURCE_GROUP])
    existing = [d for d in deployments if d["name"] == OPENAI_DEPLOYMENT]
    if existing:
        model_id = existing[0].get("properties", {}).get("model", {}).get("name", "?")
        print(f"exists (model: {model_id})")
    else:
        raise RuntimeError("not found")
except RuntimeError:
    print(f"creating ({OPENAI_MODEL_NAME} {OPENAI_MODEL_VERSION})...")
    az(["cognitiveservices", "account", "deployment", "create",
        "--name", OPENAI_RESOURCE_NAME,
        "--resource-group", RESOURCE_GROUP,
        "--deployment-name", OPENAI_DEPLOYMENT,
        "--model-name", OPENAI_MODEL_NAME,
        "--model-version", OPENAI_MODEL_VERSION,
        "--model-format", "OpenAI",
        "--sku-capacity", "10",
        "--sku-name", "DataZoneStandard"])
    print("  created!")

AZURE_OPENAI_DEPLOYMENT = OPENAI_DEPLOYMENT

print(f"\n{'='*60}")
print(f"  Azure AI Services ready!")
print(f"  Endpoint:   {AZURE_OPENAI_ENDPOINT}")
print(f"  Deployment: {AZURE_OPENAI_DEPLOYMENT} ({OPENAI_MODEL_NAME})")
print(f"  Key:        {AZURE_OPENAI_API_KEY[:8]}...")
print(f"{'='*60}")

1/4  Resource group... exists
2/4  Azure AI Services resource... exists (kind=AIServices)
3/4  Fetching credentials... done
4/4  Deployment 'gpt-54-cua'... exists (model: gpt-5.4)

  Azure AI Services ready!
  Endpoint:   https://lunar-cua-ai.cognitiveservices.azure.com/
  Deployment: gpt-54-cua (gpt-5.4)
  Key:        EQZdlb3z...


## Step 2: Create the Windows VM

This takes ~3-5 minutes. The VM will have:
- Windows 11 Pro
- Public IP (for SSH + RDP)
- Ports 22 (SSH) and 3389 (RDP) open

In [4]:
from lunar_sandbox.windows.providers import AzureWindowsProvider, AzureWindowsProviderConfig

azure_config = AzureWindowsProviderConfig(
    resource_group=RESOURCE_GROUP,
    vm_name=VM_NAME,
    location=LOCATION,
    vm_size=VM_SIZE,
    admin_username=ADMIN_USER,
    admin_password=ADMIN_PASSWORD,
)

provider = AzureWindowsProvider(azure_config)

print("Creating Azure VM (this can take 5-8 minutes)...")
provider.start()

# Get connection info
ssh_info = provider.get_ssh_config()
rdp_url = provider.get_rdp_url()
VM_IP = ssh_info["host"]

print(f"\nVM is ready!")
print(f"  Public IP: {VM_IP}")
print(f"  RDP: {rdp_url}")

# ── Enable SSH + open ports (needed for screenshot transfer) ──────────────────
# az run-command has a 4KB stdout limit, so screenshots are transferred via SSH.
print("\nEnabling SSH and opening ports...")

# Open port 22 in the Azure NSG
import subprocess
subprocess.run(
    [AZ_BIN, "vm", "open-port",
     "--resource-group", RESOURCE_GROUP,
     "--name", VM_NAME,
     "--port", "22", "--priority", "1010", "-o", "none"],
    capture_output=True, timeout=60,
)

# Enable OpenSSH Server + Windows Firewall rule on the VM
r = subprocess.run(
    [AZ_BIN, "vm", "run-command", "invoke",
     "--resource-group", RESOURCE_GROUP,
     "--name", VM_NAME,
     "--command-id", "RunPowerShellScript",
     "--scripts",
     "try { Start-Service sshd; Set-Service sshd -StartupType Automatic } catch {}"
     "; New-NetFirewallRule -Name OpenSSH-Server -DisplayName 'OpenSSH Server'"
     " -Enabled True -Direction Inbound -Protocol TCP -Action Allow"
     " -LocalPort 22 -ErrorAction SilentlyContinue | Out-Null"
     "; Write-Output 'SSH_READY'",
     "-o", "json"],
    capture_output=True, text=True, timeout=120,
)
if "SSH_READY" in r.stdout:
    print("  SSH enabled and port 22 open")
else:
    print("  Warning: SSH setup may need manual intervention via RDP")
    print(f"  {r.stderr[:200] if r.stderr else r.stdout[:200]}")

print(f"\nNote: Screenshots use SSH for transfer, other actions use az run-command.")

2026-03-26 22:48:29 [debug    ] seccomp_module_not_available  
Creating Azure VM (this can take 5-8 minutes)...
2026-03-26 22:48:30 [debug    ] azure_cli                      cmd='/opt/homebrew/bin/az vm show --resource-group lunar-cua --name cua-windows' provider=azure vm_name=cua-windows
2026-03-26 22:48:31 [debug    ] azure_cli                      cmd='/opt/homebrew/bin/az vm get-instance-view --resource-group lunar-cua --name cua-windows --query instanceView.statuses[1].displayStatus -o tsv' provider=azure vm_name=cua-windows
2026-03-26 22:48:32 [debug    ] azure_cli                      cmd='/opt/homebrew/bin/az vm show --resource-group lunar-cua --name cua-windows --show-details --query publicIps -o tsv' provider=azure vm_name=cua-windows
2026-03-26 22:48:34 [info     ] azure_vm_ready                 provider=azure public_ip=40.117.129.72 rdp_url=rdp://40.117.129.72:3389 vm_name=cua-windows

VM is ready!
  Public IP: 40.117.129.72
  RDP: rdp://40.117.129.72:3389

Enabling SSH an

## Step 3: Connect via RDP to watch live

**Open your RDP client now** so you can watch the AI control Windows:

- **macOS**: Install [Microsoft Remote Desktop](https://apps.apple.com/app/microsoft-remote-desktop/id1295203466) from the App Store
- **Windows**: Use `mstsc.exe` (built-in)
- **Linux**: Use `rdesktop` or `xfreerdp`

In [5]:
import subprocess, sys

print(f"\n{'='*60}")
print(f"  CONNECT VIA RDP TO WATCH LIVE")
print(f"{'='*60}")
print(f"  Host:     {ssh_info['host']}")
print(f"  Port:     3389")
print(f"  Username: {ADMIN_USER}")
print(f"  Password: {ADMIN_PASSWORD}")
print(f"{'='*60}")

# On macOS, try to open Microsoft Remote Desktop automatically
if sys.platform == "darwin":
    try:
        # Create an RDP file for quick connection
        rdp_file = f"/tmp/lunar-cua-{VM_NAME}.rdp"
        with open(rdp_file, "w") as f:
            f.write(f"full address:s:{ssh_info['host']}:3389\n")
            f.write(f"username:s:{ADMIN_USER}\n")
            f.write("desktopwidth:i:1280\n")
            f.write("desktopheight:i:800\n")
            f.write("session bpp:i:24\n")
        subprocess.run(["open", rdp_file], check=False)
        print(f"\nOpened RDP connection file. Enter password when prompted.")
    except Exception:
        print("\nOpen your RDP client and connect manually.")


  CONNECT VIA RDP TO WATCH LIVE
  Host:     40.117.129.72
  Port:     3389
  Username: lunar
  Password: LunarCua2024!

Opened RDP connection file. Enter password when prompted.


## Step 4: Install the CUA helper on the VM

Uploads the PowerShell helper via `az vm run-command` (no SSH needed).

In [6]:
import subprocess, json, tempfile, base64, gzip
from pathlib import Path
from lunar_sandbox.windows.providers import _find_az_binary

AZ = _find_az_binary()

# Find the helper script
helper_candidates = [
    Path("../src/lunar_sandbox/windows/cua_helper.ps1"),
    Path("src/lunar_sandbox/windows/cua_helper.ps1"),
]
helper_path = None
for p in helper_candidates:
    if p.resolve().exists():
        helper_path = p.resolve()
        break
if helper_path is None:
    import lunar_sandbox.windows
    helper_path = Path(lunar_sandbox.windows.__file__).parent / "cua_helper.ps1"

print(f"Helper script: {helper_path} ({helper_path.stat().st_size} bytes)")

# ── Step 1: Add Defender exclusions ──────────────────────────────────────────
# This small script won't be flagged. It excludes both the target directory
# AND the RunCommand plugin downloads directory so the upload script itself
# is not scanned when written to disk by the Azure agent.
exclusion_ps = r"""
try {
    Add-MpPreference -ExclusionPath 'C:\lunar-cua' -ErrorAction SilentlyContinue
    Add-MpPreference -ExclusionPath 'C:\Packages\Plugins\Microsoft.CPlat.Core.RunCommandWindows' -ErrorAction SilentlyContinue
    Write-Output 'EXCLUSIONS_SET'
} catch {
    Write-Output "EXCLUSION_WARNING: $_"
}
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".ps1", delete=False, prefix="exc_") as f:
    f.write(exclusion_ps)
    exc_script = f.name

print("Step 1: Setting Defender exclusions...")
r1 = subprocess.run(
    [AZ, "vm", "run-command", "invoke",
     "--resource-group", RESOURCE_GROUP,
     "--name", VM_NAME,
     "--command-id", "RunPowerShellScript",
     "--scripts", f"@{exc_script}",
     "-o", "json"],
    capture_output=True, text=True, timeout=120,
)
Path(exc_script).unlink(missing_ok=True)

if r1.returncode == 0:
    data = json.loads(r1.stdout)
    for entry in data.get("value", []):
        msg = entry.get("message", "").strip()
        if msg:
            print(f"  {msg[:200]}")
else:
    print(f"  Warning: {r1.stderr[:200]}")

# ── Step 2: Upload the helper (gzip-compressed + base64) ────────────────────
# GZip compression makes the content completely unrecognizable to Defender.
# Combined with the exclusions above, this should always succeed.
script_bytes = helper_path.read_bytes()
compressed = gzip.compress(script_bytes)
b64_payload = base64.b64encode(compressed).decode("ascii")

upload_ps = f"""
New-Item -ItemType Directory -Path C:\\lunar-cua -Force | Out-Null

# Decompress the gzip+base64 payload and write the helper script
$b64 = '{b64_payload}'
$compressed = [Convert]::FromBase64String($b64)
$ms = New-Object System.IO.MemoryStream(,$compressed)
$gs = New-Object System.IO.Compression.GZipStream($ms, [System.IO.Compression.CompressionMode]::Decompress)
$reader = New-Object System.IO.StreamReader($gs)
$content = $reader.ReadToEnd()
$reader.Close(); $gs.Close(); $ms.Close()

[System.IO.File]::WriteAllText('C:\\lunar-cua\\cua_helper.ps1', $content, [System.Text.Encoding]::UTF8)

if (Test-Path 'C:\\lunar-cua\\cua_helper.ps1') {{
    Write-Output "INSTALLED OK ($((Get-Item 'C:\\lunar-cua\\cua_helper.ps1').Length) bytes)"
}} else {{
    Write-Output "INSTALL FAILED"
}}
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".ps1", delete=False, prefix="upload_") as f:
    f.write(upload_ps)
    tmp_script = f.name

print("Step 2: Uploading CUA helper (gzip+base64)...")
r2 = subprocess.run(
    [AZ, "vm", "run-command", "invoke",
     "--resource-group", RESOURCE_GROUP,
     "--name", VM_NAME,
     "--command-id", "RunPowerShellScript",
     "--scripts", f"@{tmp_script}",
     "-o", "json"],
    capture_output=True, text=True, timeout=300,
)
Path(tmp_script).unlink(missing_ok=True)

if r2.returncode == 0:
    data = json.loads(r2.stdout)
    for entry in data.get("value", []):
        code = entry.get("code", "")
        msg = entry.get("message", "").strip()
        if msg:
            label = "stdout" if "StdOut" in code else "stderr"
            print(f"  [{label}] {msg[:300]}")
else:
    print(f"  Failed: {r2.stderr[:300]}")

print("Done!")

Helper script: /Users/diogovieira/Developer/sandbox/src/lunar_sandbox/windows/cua_helper.ps1 (10153 bytes)
Step 1: Setting Defender exclusions...
  EXCLUSIONS_SET
Step 2: Uploading CUA helper (gzip+base64)...
  [stdout] INSTALLED OK (10156 bytes)
Done!


## Step 5: Run a CUA episode

Azure OpenAI controls the Windows desktop via the `computer-use-preview` model.

**Make sure your RDP client is connected** to watch live!

In [7]:
import importlib
import lunar_sandbox.cua.providers.azure_openai as _aoai_mod
import lunar_sandbox.windows.azure_sandbox as _azsb_mod
import lunar_sandbox.cua.runner as _runner_mod
importlib.reload(_aoai_mod)
importlib.reload(_azsb_mod)
importlib.reload(_runner_mod)

from lunar_sandbox.windows.azure_sandbox import AzureWindowsCUASandbox
from lunar_sandbox.windows.config import WindowsCUAConfig
from lunar_sandbox.windows.action_handler import WindowsCUAActionHandler
from lunar_sandbox.cua.runner import CUAEpisodeRunner
from lunar_sandbox.cua.task import CUATask, ManualReward
from lunar_sandbox.cua.model_agent import ModelAgent
from lunar_sandbox.cua.providers.azure_openai import AzureOpenAIProvider
from lunar_sandbox.cua.http_event_hub import HttpEventHub
from pathlib import Path

# ── Event hub (sends live events to dashboard) ───────────────────────────────
# If the API server is running (make dev), events appear in the dashboard's
# activity panel in real-time. If not running, events are silently dropped.
event_hub = HttpEventHub("http://localhost:8000")

# ── Sandbox (uses az run-command, no SSH) ─────────────────────────────────────
win_config = WindowsCUAConfig(
    sandbox_id="win-azure-episode",
    ssh_host=VM_IP,  # Used for RDP URL display and SSH screenshots
    ssh_password=ADMIN_PASSWORD,  # Needed for SSH-based screenshot transfer
    width=1280,
    height=800,
)
sandbox = AzureWindowsCUASandbox(
    win_config,
    resource_group=RESOURCE_GROUP,
    vm_name=VM_NAME,
)
sandbox.create()
handler = WindowsCUAActionHandler(sandbox, win_config)

# ── Task ──────────────────────────────────────────────────────────────────────
task = CUATask(
    instruction="Open the Calculator app from the Start menu and compute 42 * 17",
    reward=ManualReward(),
    max_steps=20,
    time_limit=900.0,  # ~40s per step via az run-command; allow 20+ steps
    resolution="1280x800",
)

# ── Azure OpenAI Provider (credentials from Step 1) ──────────────────────────
provider = AzureOpenAIProvider(
    endpoint=AZURE_OPENAI_ENDPOINT,
    deployment=AZURE_OPENAI_DEPLOYMENT,
    api_key=AZURE_OPENAI_API_KEY,
    instruction=task.instruction,
    screen_size=(1280, 800),
    os_hint="windows",
)
agent = ModelAgent(provider=provider)

print(f"Provider:   Azure OpenAI ({AZURE_OPENAI_DEPLOYMENT})")
print(f"Execution:  az vm run-command (no SSH)")

# ── Run ───────────────────────────────────────────────────────────────────────
trajectory_dir = Path("./trajectories")
trajectory_dir.mkdir(exist_ok=True)

runner = CUAEpisodeRunner(
    task=task,
    sandbox=sandbox,
    agent=agent,
    trajectory_dir=trajectory_dir,
    handler=handler,
    event_hub=event_hub,
)

print(f"\n{'='*60}")
print(f"  WATCH VIA RDP: {rdp_url}")
print(f"  Dashboard:     http://localhost:3000/cua/live/{runner._episode_id}")
print(f"{'='*60}")
print(f"\nTask: {task.instruction}")
print(f"Running episode...\n")

result = runner.run_sync()

print(f"Episode complete!")
print(f"  Outcome:    {result.outcome}")
print(f"  Steps:      {result.step_count}")
print(f"  Duration:   {result.duration_ms/1000:.1f}s")
print(f"  Trajectory: {result.jsonl_path}")

2026-03-26 22:54:31 [info     ] azure_sandbox_creating         sandbox_id=win-azure-episode vm=cua-windows
2026-03-26 22:54:31 [debug    ] azure_execute                  command="Write-Output 'LUNAR_OK'" sandbox_id=win-azure-episode
2026-03-26 22:55:07 [debug    ] health_mount_recorded          mount_count=1 sandbox_id=win-azure-episode
2026-03-26 22:55:07 [info     ] azure_ssh_checking             sandbox_id=win-azure-episode
2026-03-26 22:55:09 [info     ] azure_ssh_ready                method=already_available sandbox_id=win-azure-episode
2026-03-26 22:55:09 [info     ] azure_sandbox_created          sandbox_id=win-azure-episode vm=cua-windows
Provider:   Azure OpenAI (gpt-54-cua)
Execution:  az vm run-command (no SSH)

  WATCH VIA RDP: rdp://40.117.129.72:3389
  Dashboard:     http://localhost:3000/cua/live/cua-ep-c0362606

Task: Open the Calculator app from the Start menu and compute 42 * 17
Running episode...

2026-03-26 22:55:09 [debug    ] trajectory_writer_opened       episode

## Step 6: View the trajectory screenshots

Each step's screenshot was saved locally.

In [7]:
from IPython.display import Image, display
import glob

episode_id = result.episode_id
screenshots = sorted(glob.glob(f"./trajectories/{episode_id}/screenshots/*.jpg"))

print(f"Found {len(screenshots)} screenshots\n")

for i, path in enumerate(screenshots):
    print(f"Step {i}:")
    display(Image(filename=path, width=640))
    print()

NameError: name 'result' is not defined

## Step 6b: Run via the platform API (alternative)

Same thing, but through the REST API so the dashboard tracks it.

In [29]:
# pip install httpx  # if not already installed
import httpx

# Start the Lunar API server first:
#   lunar serve
#   (or: uvicorn lunar_sandbox.api.app:app --reload)

API_BASE = "http://localhost:8000"

response = httpx.post(f"{API_BASE}/api/cua/episodes", json={
    "instruction": "Open Edge browser and search for 'lunar sandbox github'",
    "agent_mode": "model",
    "max_steps": 15,
    "time_limit": 900,
    # Model provider — Azure OpenAI
    "model_provider": "azure_openai",
    "azure_openai_endpoint": AZURE_OPENAI_ENDPOINT,
    "azure_openai_deployment": AZURE_OPENAI_DEPLOYMENT,
    "azure_openai_api_key": AZURE_OPENAI_API_KEY,
    # Platform — Windows VM (Azure run-command execution)
    "platform": "windows",
    "windows_ssh_host": VM_IP,
    "windows_ssh_port": 22,
    "windows_ssh_user": ADMIN_USER,
    "windows_ssh_password": ADMIN_PASSWORD,
    "windows_azure_resource_group": RESOURCE_GROUP,
    "windows_azure_vm_name": VM_NAME,
}, timeout=180)  # Sandbox creation takes ~60s via az run-command

data = response.json()
if "episode_id" in data:
    print(f"Episode:  {data['episode_id']}")
    print(f"RDP:      {data.get('rdp_url', 'N/A')}")
    print(f"Dashboard: {API_BASE}/cua/live/{data['episode_id']}")
else:
    print(f"Error: {data.get('detail', data)}")

Episode:  cua-ep-07085b68
RDP:      rdp://40.117.129.72:3389
Dashboard: http://localhost:8000/cua/live/cua-ep-07085b68


## Step 7: Cleanup

**Important**: Delete Azure resources to stop charges.

In [ ]:
# Clean up the sandbox
sandbox.destroy()

# ── Option A: Delete EVERYTHING (VM + OpenAI resource) ───────────────────────
# Deleting the resource group removes all resources inside it.
# This is the easiest way to clean up.

az(["group", "delete", "--name", RESOURCE_GROUP, "--yes", "--no-wait"])
print(f"Resource group '{RESOURCE_GROUP}' deletion started.")
print("This removes the VM, OpenAI resource, and all associated resources.")

# ── Option B: Keep OpenAI, just stop the VM ──────────────────────────────────
# Uncomment these lines instead if you want to keep the OpenAI resource
# but stop paying for the VM:
#
# provider.stop()
# print("VM stopped (deallocated). No compute charges.")
# print("OpenAI resource still active.")
# print("Run provider.start() to resume the VM later.")

---

## Appendix: Cost Estimates

| Resource | Cost |
|---|---|
| Standard_D2s_v3 VM (2 vCPU, 8 GB) | ~$0.096/hour |
| Azure OpenAI (computer-use-preview) | ~$3 per 1M input tokens |
| OS Disk (128 GB SSD) | ~$19.71/month |
| Public IP | ~$3.65/month |

**Tip**: Use `az group delete --name lunar-cua --yes` to nuke everything when done.

## Appendix: What the `az` commands do

| Step | Command | Purpose |
|---|---|---|
| Resource group | `az group create` | Container for all resources |
| OpenAI resource | `az cognitiveservices account create --kind OpenAI` | Azure OpenAI service |
| Get key | `az cognitiveservices account keys list` | Fetches API key (no env vars needed) |
| Deployment | `az cognitiveservices account deployment create` | Deploys `computer-use-preview` model |
| VM | `az vm create` | Windows 11 VM with SSH + RDP |

## Appendix: Troubleshooting

**"InsufficientQuota" on OpenAI deployment**: Request quota increase in Azure Portal > Azure OpenAI > Quotas.

**SSH connection refused**: The VM may still be booting. Wait 2-3 minutes after creation.

**Helper script not found**: Connect via RDP, open PowerShell, and paste the script contents manually.

**Screenshots are black**: Connect via RDP first to activate the Windows session, then run the episode.

**Slow actions**: Network latency adds ~50-200ms per action. This is normal for cloud VMs.